In [ ]:
pip install openai

In [1]:
from openai import OpenAI
from dotenv import load_dotenv
import os
import time
import pandas as pd
from openai import RateLimitError
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report

In [2]:
# Loading API key through a .env file. The .env file is not uploaded to github.

load_dotenv('the_nlp.env')  
api_key = os.getenv("OPENROUTER_API_KEY")                 # With credit

client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=api_key)

In [3]:
# Importing and checking speeches
# Potential for function here

h5 = pd.read_csv('hansard500.csv')

print(h5.shape, '\n')
print(h5.isnull().sum(), '\n')
print(h5['speech_class'].value_counts(), '\n')
print(h5['party'].value_counts(), '\n')

h5.head()

(500, 8) 

speech            0
party            19
constituency     19
date              0
speech_class      0
major_heading     0
year              0
speakername       0
dtype: int64 

speech_class
Speech        481
Procedural     16
Division        3
Name: count, dtype: int64 

party
Conservative                        313
Labour                               88
Scottish National Party              32
Labour (Co-op)                       19
Speaker                              11
Democratic Unionist Party             7
Liberal Democrat                      5
Plaid Cymru                           3
Independent                           1
Social Democratic & Labour Party      1
Green Party                           1
Name: count, dtype: int64 



,speech,party,constituency,date,speech_class,major_heading,year,speakername
0,We will now suspend for three minutes for sani...,Conservative,Ribble Valley,2021-03-11,Speech,Contingencies Fund (No. 2) Bill,2021,Nigel Evans
1,I am now beginning to share the indignation of...,Labour,City of Chester,2020-11-24,Speech,Exiting the European Union,2020,Christian Matheson
2,"The hon. Lady is wrong; as a matter of fact, w...",Conservative,Newark,2021-02-10,Speech,Building Safety,2021,Robert Jenrick
3,In order to allow the safe exit of hon. Member...,Speaker,Chorley,2020-10-07,Speech,Prime Minister,2020,Lindsay Hoyle
4,Welsh lamb is a premium product that is wanted...,Conservative,Montgomeryshire,2021-02-03,Speech,Wales,2021,Craig Williams


In [4]:
# Make a note that I would take out Lib Dems because of such low nr. of examples if I were doing this professionally

h5['party'] = h5['party'].replace('Labour (Co-op)', 'Labour')
h5 = h5[h5['speech_class'].isin(['Speech'])]
h5 = h5[h5['party'].isin(['Conservative', 'Labour', 'Scottish National Party', 'Liberal Democrat'])]

print(h5['speech_class'].value_counts(), '\n')
print(h5['party'].value_counts(), '\n')

h5.head()

speech_class
Speech    457
Name: count, dtype: int64 

party
Conservative               313
Labour                     107
Scottish National Party     32
Liberal Democrat             5
Name: count, dtype: int64 



,speech,party,constituency,date,speech_class,major_heading,year,speakername
0,We will now suspend for three minutes for sani...,Conservative,Ribble Valley,2021-03-11,Speech,Contingencies Fund (No. 2) Bill,2021,Nigel Evans
1,I am now beginning to share the indignation of...,Labour,City of Chester,2020-11-24,Speech,Exiting the European Union,2020,Christian Matheson
2,"The hon. Lady is wrong; as a matter of fact, w...",Conservative,Newark,2021-02-10,Speech,Building Safety,2021,Robert Jenrick
4,Welsh lamb is a premium product that is wanted...,Conservative,Montgomeryshire,2021-02-03,Speech,Wales,2021,Craig Williams
5,"Given that this was outlined earlier today, it...",Conservative,Great Yarmouth,2021-04-13,Speech,Northern Ireland,2021,Brandon Lewis


In [5]:
x = h5['speech']          
y = h5['party']

x_train, x_test, y_train, y_test = train_test_split(
    x, y, stratify=y, random_state=26)                    # Same seed as part 2 for consistency

labels = sorted(y.unique().tolist())                      # Creating a list of party labels to feed into LLM
label_str = ", ".join(labels)

In [6]:
# function to talk to LLM
# Built in rate limit error catcher

def classify(prompt, max_retries=5):
    for attempt in range(max_retries):
        try:
            resp = client.chat.completions.create(
                model="meta-llama/llama-3.3-70b-instruct",
                messages=[{"role": "user", "content": prompt}],
                temperature=0,                                     # Temperature set at 0 as I want the LLM to be as unimaginative as possible
                max_tokens=10,                                     # Limiting tokens to limit answer, we only want a single label
            )
            return resp.choices[0].message.content.strip()
        except RateLimitError:
            wait = 30 * (attempt + 1)   
            print(f"Rate-limited, waiting {wait}s (attempt {attempt+1}/{max_retries})")
            time.sleep(wait)
    raise RuntimeError("Still rate-limited after retries")

In [7]:
def normalize(out):
    '''This is a function to normalize output from the LLM and maintain consistency'''
    out = out.lower()
    for lab in labels:
        if lab.lower() in out:
            return lab
    return "Labour"   

In [8]:
print(classify("Reply with one word: hello"))

Hello


In [9]:
ZERO_SHOT_PROMPT = """Classify UK parliamentary speeches by the speaker's political party.
Read the speech and respond with exactly one label from this list: ['Conservative', 'Labour', 'Scottish National Party', 'Liberal Democrat']
Respond with the label only. Do not add an explanation or punctuation. There should be nothing beyond the label.

Speech:
{speech}

Party:"""

In [16]:
FEW_SHOT_PROMPT = """Classify UK parliamentary speeches by the speaker's political party.
Read the speech and respond with exactly one label from this list: ['Conservative', 'Labour', 'Scottish National Party', 'Liberal Democrat']
Respond with the label only. Do not add an explanation or punctuation. There should be nothing beyond the label.

Here are some labelled examples:

{examples}

Now classify this speech. Respond with the label only.

Speech:
{speech}

Party:"""

In [18]:
def build_examples(n_per_party=1):
    '''Function to build some examples for the few shot process. All taken from the training data'''
    shots = []
    for lab in labels:
        idx = y_train[y_train == lab].index[:n_per_party]
        for i in idx:
            shots.append((x_train.loc[i][:600], lab))
    return shots

In [19]:
example_block = "\n\n".join(
    f"Speech: {sp}\nParty: {lab}" for sp, lab in build_examples())

In [10]:
def run(prompt_template, few_shot=False):
    preds = []
    for i, s in enumerate(x_test):
        if few_shot:
            p = prompt_template.format(examples=example_block, speech=s[:3000])
        else:
            p = prompt_template.format(speech=s[:4000])
        preds.append(normalize(classify(p)))
        if (i + 1) % 10 == 0:
            print(f"  {i+1}/{len(x_test)} done")
        time.sleep(3.5)                                     # stay under 20 req/min to regulate usage
    return preds

In [11]:
def score(name, preds):
    print(f"\n===== {name} =====")
    print("Macro-avg F1:", f1_score(y_test, preds, average='macro', zero_division=0))
    print(classification_report(y_test, preds, zero_division=0))

In [13]:
# Making sure everything is connected properly before putting the whole list of speeches through the LLM

print("Smoke test (zero-shot, 5 speeches):")
for s in x_test[:5]:
    print(" ->", normalize(classify(ZERO_SHOT_PROMPT.format(labels=label_str, speech=s[:4000]))))      # Speeches limited at 4000 characters for consistency and usage

Smoke test (zero-shot, 5 speeches):
 -> Conservative
 -> Conservative
 -> Conservative
 -> Conservative
 -> Conservative


In [14]:
zero_preds = run(ZERO_SHOT_PROMPT, few_shot=False)
score("ZERO-SHOT", zero_preds)


  10/115 done
  20/115 done
  30/115 done
  40/115 done
  50/115 done
  60/115 done
  70/115 done
  80/115 done
  90/115 done
  100/115 done
  110/115 done

===== ZERO-SHOT =====
Macro-avg F1: 0.527363184079602
                         precision    recall  f1-score   support

           Conservative       0.99      0.89      0.93        79
                 Labour       0.65      0.96      0.78        27
       Liberal Democrat       0.00      0.00      0.00         1
Scottish National Party       1.00      0.25      0.40         8

               accuracy                           0.85       115
              macro avg       0.66      0.52      0.53       115
           weighted avg       0.90      0.85      0.85       115



In [20]:
few_preds  = run(FEW_SHOT_PROMPT, few_shot=True)
score("FEW-SHOT", few_preds)

  10/115 done
  20/115 done
  30/115 done
  40/115 done
  50/115 done
  60/115 done
  70/115 done
  80/115 done
  90/115 done
  100/115 done
  110/115 done

===== FEW-SHOT =====
Macro-avg F1: 0.58438995215311
                         precision    recall  f1-score   support

           Conservative       0.94      0.96      0.95        79
                 Labour       0.80      0.89      0.84        27
       Liberal Democrat       0.00      0.00      0.00         1
Scottish National Party       1.00      0.38      0.55         8

               accuracy                           0.90       115
              macro avg       0.68      0.56      0.58       115
           weighted avg       0.90      0.90      0.89       115

